In [ ]:
import os
import inspect
import numpy as np
import pandas as pd
import torch
import gradio as gr

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    pipeline,
    DataCollatorWithPadding
)

device = "cuda" if torch.cuda.is_available() else "cpu"
pipe_device = 0 if torch.cuda.is_available() else -1



BASE_MODEL = "google/muril-base-cased"


# Loading Data


df_sent = pd.read_csv("sentiment_data.csv")
df_hate = pd.read_csv("hate_speech.csv")
df_sarc = pd.read_csv("sarcasm_dataset.csv")

df_sent = df_sent.rename(columns={'sentence': 'text'})
df_hate = df_hate.rename(columns={'new_label': 'label'})


# Prepare Data

def prepare_and_split(df, task_name):

    if "text" not in df.columns or "label" not in df.columns:
        raise ValueError(f"{task_name}")

    df = df[["text", "label"]].copy()

    df["text"] = df["text"].astype(str).str.lower().str.strip()
    df["label"] = df["label"].astype(int)

    train_df, test_df = train_test_split(
        df,
        test_size=0.30,
        random_state=42,
        stratify=df["label"]
    )

    print(
        f"{task_name} Split -> "
        f"Train: {len(train_df)}, Test: {len(test_df)}"
    )

    return train_df, test_df


train_s, test_s = prepare_and_split(df_sent, "Sentiment")
train_h, test_h = prepare_and_split(df_hate, "Hate Speech")
train_r, test_r = prepare_and_split(df_sarc, "Sarcasm")


# Metrics

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted")
    }


# Version-Safe TrainingArguments

def make_training_args(output_path, task_name):
    args = {
        "output_dir": output_path,
        "num_train_epochs": 3,
        "per_device_train_batch_size": 8,
        "per_device_eval_batch_size": 8,
        "learning_rate": 2e-5,
        "report_to": "none"
    }

    sig = inspect.signature(TrainingArguments.__init__)

    return TrainingArguments(**args)


# Fine-Tuning Function

def fine_tune_task(train_df, task_name, num_labels, id2label, label2id):
    print(f"\nFine-tuning {task_name} model...")

    output_path = f"./models/{task_name}"

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

    train_ds = Dataset.from_pandas(train_df, preserve_index=False)

    def tokenize(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            max_length=128
        )

    train_ds = train_ds.map(tokenize, batched=True)

    train_ds = train_ds.rename_column("label", "labels")

    train_ds = train_ds.remove_columns(["text"])

    model = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id
    )

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    training_args = make_training_args(output_path, task_name)

    trainer_kwargs = {
        "model": model,
        "args": training_args,
        "train_dataset": train_ds,
        "data_collator": data_collator,
        "compute_metrics": compute_metrics
    }

    trainer_sig = inspect.signature(Trainer.__init__)

    if "processing_class" in trainer_sig.parameters:
        trainer_kwargs["processing_class"] = tokenizer
    else:
        trainer_kwargs["tokenizer"] = tokenizer

    trainer = Trainer(**trainer_kwargs)

    trainer.train()

    trainer.save_model(output_path)
    tokenizer.save_pretrained(output_path)

    print(f"{task_name} model saved at {output_path}")

    return output_path


# Label Mappings

sent_id2label = {
    0: "negative",
    1: "neutral",
    2: "positive"
}
sent_label2id = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

hate_id2label = {
    0: "clean",
    1: "abusive"
}
hate_label2id = {
    "clean": 0,
    "abusive": 1
}

sarc_id2label = {
    0: "literal",
    1: "sarcastic"
}
sarc_label2id = {
    "literal": 0,
    "sarcastic": 1
}


# Train Models

TRAIN_MODELS = True

if TRAIN_MODELS:
    sent_model_path = fine_tune_task(
        train_s,
        "sentiment",
        3,
        sent_id2label,
        sent_label2id
    )
    hate_model_path = fine_tune_task(
        train_h,
        "hate_speech",
        2,
        hate_id2label,
        hate_label2id
    )

    sarc_model_path = fine_tune_task(
        train_r,
        "sarcasm",
        2,
        sarc_id2label,
        sarc_label2id
    )
else:
    sent_model_path = "./models/sentiment"
    hate_model_path = "./models/hate_speech"
    sarc_model_path = "./models/sarcasm"


# Load Trained Pipelines


base_tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    use_fast=False
)

sent_pipe = pipeline(
    "text-classification",
    model="./models/sentiment",
    tokenizer=base_tokenizer,
    device=pipe_device
)

hate_pipe = pipeline(
    "text-classification",
    model="./models/hate_speech",
    tokenizer=base_tokenizer,
    device=pipe_device
)

sarc_pipe = pipeline(
    "text-classification",
    model="./models/sarcasm",
    tokenizer=base_tokenizer,
    device=pipe_device
)



# Evaluation

def evaluate_model(test_df, pipe_func, task_name, label2id):
    texts = test_df["text"].tolist()
    y_true = test_df["label"].tolist()

    preds = pipe_func(
        texts,
        truncation=True,
        max_length=128,
        batch_size=16
    )

    y_pred = [label2id[p["label"]] for p in preds]

    print(f"\nClassification Report: {task_name}")
    print(classification_report(y_true, y_pred))

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="weighted")

    print(f"{task_name} Accuracy: {acc:.4f}")
    print(f"{task_name} F1 Score: {f1:.4f}")


evaluate_model(test_s, sent_pipe, "Sentiment", sent_label2id)
evaluate_model(test_h, hate_pipe, "Hate Speech", hate_label2id)
evaluate_model(test_r, sarc_pipe, "Sarcasm", sarc_label2id)


# Final App Function

def final_pipeline(text):
    if text is None or text.strip() == "":
        return "N/A", "N/A", "N/A"

    text = text.lower().strip()

    sent_result = sent_pipe(text)[0]
    hate_result = hate_pipe(text)[0]
    sarc_result = sarc_pipe(text)[0]

    sentiment = sent_result["label"]
    hate = hate_result["label"]
    sarcasm = sarc_result["label"]

    return sentiment, hate, sarcasm


# Gradio UI

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# IIT Term Paper: Multi-Task Hinglish NLU")
    gr.Markdown("Sentiment, hate speech, and sarcasm detection using fine-tuned MuRIL models.")

    user_input = gr.Textbox(
        label="Hinglish Input",

    )

    with gr.Row():
        out_sent = gr.Label(label="Sentiment")
        out_hate = gr.Label(label="Hate Speech")
        out_sarc = gr.Label(label="Sarcasm")

    btn = gr.Button("Analyze Text", variant="primary")

    btn.click(
        fn=final_pipeline,
        inputs=user_input,
        outputs=[ out_sent, out_hate, out_sarc]
    )

demo.launch()

In [ ]:
import shutil
import os
from google.colab import files


sentiment_model_dir = './models/sentiment'
hate_speech_model_dir = './models/hate_speech'
sarcasm_model_dir = './models/sarcasm'

zip_filename = 'models.zip'

temp_zip_dir = './all_models_to_zip'
os.makedirs(temp_zip_dir, exist_ok=True)

shutil.copytree(sentiment_model_dir, os.path.join(temp_zip_dir, 'sentiment'), dirs_exist_ok=True)
shutil.copytree(hate_speech_model_dir, os.path.join(temp_zip_dir, 'hate_speech'), dirs_exist_ok=True)
shutil.copytree(sarcasm_model_dir, os.path.join(temp_zip_dir, 'sarcasm'), dirs_exist_ok=True)

shutil.make_archive(base_name='models', format='zip', root_dir='.', base_dir=temp_zip_dir)
print(f"Created {zip_filename} containing both models.")
files.download(zip_filename)

shutil.rmtree(temp_zip_dir)
print("Temporary files cleaned up.")

In [ ]:
pip install pandas numpy torch scikit-learn transformers matplotlib

In [ ]:
# Evaluation

import os
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np
import pandas as pd

# Defining global variables

if 'DEVICE' not in globals():
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if 'OUTPUT_DIR' not in globals():
    OUTPUT_DIR = "./evaluation_reports"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

def evaluate_model(task_name, model_path, test_df, label_names, batch_size=16):
    print(f"\nEvaluating {task_name}...")

    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model folder not found: {model_path}")

    tokenizer = AutoTokenizer.from_pretrained(model_path)

    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    model.to(DEVICE)
    model.eval()

    texts = test_df["text"].tolist()
    y_true = test_df["label"].tolist()

    all_preds = []

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]

        inputs = tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=128,
            return_tensors="pt"
        )

        inputs = {key: value.to(DEVICE) for key, value in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        all_preds.extend(preds)

    y_pred = np.array(all_preds)

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="weighted")

    report = classification_report(
        y_true,
        y_pred,
        labels=list(range(len(label_names))),
        target_names=label_names,
        zero_division=0
    )

    print(f"{task_name} Accuracy: {acc:.4f}")
    print(f"{task_name} F1 Score: {f1:.4f}")
    print(report)

    report_path = os.path.join(OUTPUT_DIR, f"{task_name}_classification_report.txt")

    with open(report_path, "w", encoding="utf-8") as f:
        f.write(f"{task_name} Evaluation Report\n")
        f.write(f"Accuracy: {acc:.4f}\n")
        f.write(f"Weighted F1 Score: {f1:.4f}\n\n")
        f.write(report)

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(len(label_names)))
    )

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=label_names
    )

    fig, ax = plt.subplots(figsize=(6, 5))
    disp.plot(
        ax=ax,
        cmap="Blues",
        values_format="d",
        colorbar=True
    )

    plt.title(f"{task_name} Confusion Matrix")
    plt.tight_layout()

    cm_path = os.path.join(OUTPUT_DIR, f"{task_name}_confusion_matrix.png")
    plt.savefig(cm_path, dpi=300)
    plt.close()

    print(f"Saved report: {report_path}")
    print(f"Saved confusion matrix: {cm_path}")

    return {
        "task": task_name,
        "accuracy": acc,
        "f1": f1
    }


# Evaluation

results = []

results.append(
    evaluate_model(
        task_name="sentiment",
        model_path="/content/models/sentiment",
        test_df=test_s,
        label_names=["negative", "neutral", "positive"]
    )
)

results.append(
    evaluate_model(
        task_name="hate_speech",
        model_path="/content/models/hate_speech",
        test_df=test_h,
        label_names=["clean", "abusive"]
    )
)

results.append(
    evaluate_model(
        task_name="sarcasm",
        model_path="/content/models/sarcasm",
        test_df=test_r,
        label_names=["literal", "sarcastic"]
    )
)


# Saving Summary

summary_df = pd.DataFrame(results)
summary_path = os.path.join(OUTPUT_DIR, "metrics_summary.csv")
summary_df.to_csv(summary_path, index=False)

print("\nAll evaluations completed.")
print(f"Summary saved: {summary_path}")